# Emissions processing pipeline

This pipeline fetches GHG emissions data from various data sources for all 
datasets in `input/config_country_datasets.csv` and generates emissions output CSVs for the datasets
that will be used in ETSource. Consult the [README.md](README.md) for more info about the data
sources and processing steps.

## 0. Configuration

### 0.1 Imports and paths

In [ ]:
import json
from pathlib import Path

import pandas as pd
import unfccc_di_api
import shutil

from lib import config, sources, processors, validators, writers

NOTEBOOK_DIR        = Path(".").resolve()
REPO_ROOT           = NOTEBOOK_DIR.parent.parent

# ── Data directories (outside tools/) ────────────────────────────────────────
DATA_DIR                    = REPO_ROOT / "data"
SOURCE_ANALYSES_DIR         = REPO_ROOT / "source_analyses"
TOOLS_EMISSIONS_INPUT_DIR   = NOTEBOOK_DIR / "input"

# ── Tool configuration (stays in tools/) ─────────────────────────────────────
DATASETS_CONFIG         = TOOLS_EMISSIONS_INPUT_DIR / "config_country_datasets.csv"

DATA_DIR.mkdir(exist_ok=True)
TOOLS_EMISSIONS_INPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Repo root    : {REPO_ROOT}")
print(f"Data dir     : {DATA_DIR}")
print(f"Source anal. : {SOURCE_ANALYSES_DIR}")
print(f"Input dir    : {TOOLS_EMISSIONS_INPUT_DIR}")

### 0.2 Dataset configuration

Setup of datasets and data source routing.

In [ ]:
# NL datasets
NL_DATASETS = {"nl", "nl2019", "nl2023"}

# Datasets skipped for specific approach UK, GB, UKNI
SKIP_DATASETS = {"GB_great_britain", "UKNI01_northern_ireland"}

# ── UNFCCC CRT Excel datasets ─────────────────────────────────────────────────
def _crt(dataset: str, year: int, filename: str) -> Path:
    return SOURCE_ANALYSES_DIR / dataset / str(year) / "15_emissions" / filename

UNFCCC_EXCEL_DATASETS: dict[str, list[tuple[int, Path]]] = {
    "RS_serbia": [
        (1990, _crt("RS_serbia", 1990, "SRB-CRT-2024-V0.2-1990-20241217-133518_started.xlsx")),
        (2019, _crt("RS_serbia", 2019, "SRB-CRT-2024-V0.2-2019-20241217-134631_started.xlsx")),
    ],
    "SG_singapore": [
        # Note: Singapore uses 2000 data for 1990 base year (oldest available data)
        (1990, _crt("SG_singapore", 2000, "SGP-CRT-2024-V0.3-2000-20241108-153202_started.xlsx")),
        (2019, _crt("SG_singapore", 2019, "SGP-CRT-2024-V0.3-2019-20241108-153202_started.xlsx")),
    ],
    "nl2023": [
        (2023, _crt("nl", 2023, "NLD-CRT-2025-V0.9-2023.xlsx")),
    ],
}

## 1. Retrieve and save raw data files

In [ ]:
# Load datasets config
datasets_df = config.load_datasets(DATASETS_CONFIG)

# UNFCCC API datasets: derived from unfccc_code column in config
UNFCCC_API_DATASETS: dict[str, str] = (
    datasets_df[datasets_df["unfccc_code"].notna() & (datasets_df["unfccc_code"] != "")]
    .set_index("dataset")["unfccc_code"]
    .to_dict()
)

print(f"Datasets to process : {len(datasets_df)}")
print(f"UNFCCC API datasets : {len(UNFCCC_API_DATASETS)}")

### 1.1 UNFCCC API source
Download and load from UNFCCC API source (for all country datasets except RS, SG, nl2023, and GB/UKNI).

#### Load and save data

In [ ]:
# ── UNFCCC API: load ZenodoReader and fetch raw data ────────────────────
print("Loading UNFCCC ZenodoReader...")
unfccc_reader = unfccc_di_api.ZenodoReader()
unfccc_available = set(unfccc_reader.parties)
print("Ready.\n")

# Initialize UNFCCC API source
api_source = sources.UFCCCAPISource(unfccc_reader)
unfccc_processed_data: dict[str, list[dict]] = {}

for _, row in datasets_df[datasets_df["dataset"].isin(UNFCCC_API_DATASETS)].iterrows():
    dataset         = row["dataset"]
    analysis_year   = int(row["analysis_year"])
    country_name    = row["name"]
    party_code      = UNFCCC_API_DATASETS[dataset]
    years           = sorted({1990, analysis_year})

    if party_code not in unfccc_available:
        print(f"  Warning: {party_code} not in UNFCCC data")
        continue

    print(f"  UNFCCC {party_code} ({dataset}): {years}")
    try:
        rows = []
        for record in api_source.fetch(dataset, years, party_code=party_code):
            # Add metadata
            record["dataset"] = dataset
            record["country_name"] = country_name
            record["analysis_year"] = analysis_year
            record["data_year"] = record["year"]
            rows.append(record)

        unfccc_processed_data[dataset] = rows
    except Exception as exc:
        print(f"  Error: {exc}")

print(f"\nFetched {len(unfccc_processed_data)} UNFCCC API dataset(s).")

### 1.2 Northern Ireland source

Emission data for Northern Ireland is fetched from downloaded excel for UK. See the [README](README.md#data-source-hierarchy) for the data source. Raw input emission file is exported as CSV to source_analyses folder. 

In [ ]:
# ── Northern Ireland: load and filter DA_GHGI spreadsheet ─────────────────
DATASET_UKNI = "UKNI01_northern_ireland"
ANALYSIS_YEAR_UKNI = int(
    datasets_df.loc[datasets_df["dataset"] == DATASET_UKNI, "analysis_year"].iloc[0]
)
YEARS_UKNI = [1990, ANALYSIS_YEAR_UKNI]

ukni_out = (
    SOURCE_ANALYSES_DIR
    / DATASET_UKNI
    / str(ANALYSIS_YEAR_UKNI)
    / "15_emissions"
    / f"input_emissions_{DATASET_UKNI}.csv"
)

if ukni_out.exists():
    print(f"Skipping — output already exists: {ukni_out.relative_to(REPO_ROOT)}")
else:
    ukni_src = (
        SOURCE_ANALYSES_DIR
        / DATASET_UKNI
        / str(ANALYSIS_YEAR_UKNI)
        / "15_emissions"
        / "DA_GHGI_1990_2023.xlsx"
    )

    ukni_df = pd.read_excel(
        ukni_src, sheet_name="By Source data", skiprows=5, header=0, dtype={"CRT_Category": str}
    )

    years_str = [str(y) for y in YEARS_UKNI]
    ukni_df = ukni_df[
        (ukni_df["RegionName"] == "Northern Ireland") & (ukni_df["EmissionYear"].isin(years_str))
    ].drop(columns=["Unnamed: 0", "Timestamp"])

    ukni_df.to_csv(ukni_out, index=False)
    print(f"Saved {len(ukni_df)} rows → {ukni_out.relative_to(REPO_ROOT)}")

## 2. Pre-process data and save intermediate files

### 2.1 UNFCCC API source

In [ ]:
# Collect all processed rows
all_rows: list[dict] = []

for dataset, rows in unfccc_processed_data.items():
    all_rows.extend(rows)

print(f"Collecting data from UNFCCC API done.")

### 2.2 UNFCCC CRT Excel source (RS, SG, nl2023)

In [ ]:
# ── UNFCCC CRT Excel datasets ──────────────────────────────────────────────
excel_source = sources.UFCCCExcelSource()

for dataset, files in UNFCCC_EXCEL_DATASETS.items():
    meta = datasets_df[datasets_df["dataset"] == dataset].iloc[0]
    country_name = meta["name"]
    analysis_year = int(meta["analysis_year"])

    for data_year, path in files:
        print(f"  CRT {dataset} ({country_name}): {data_year} ← {path.name}")
        try:
            for record in excel_source.fetch(dataset, [data_year], excel_path=path):
                record["dataset"] = dataset
                record["country_name"] = country_name
                record["analysis_year"] = analysis_year
                record["data_year"] = data_year
                all_rows.append(record)
        except Exception as exc:
            print(f"  Error: {exc}")

### 2.3 Northern Ireland: normalize DA_GHGI to CRT format

In [ ]:
# ── Northern Ireland: normalize DA_GHGI to CRT format ─────────────────────
ukni_raw = (
    SOURCE_ANALYSES_DIR
    / DATASET_UKNI
    / str(ANALYSIS_YEAR_UKNI)
    / "15_emissions"
    / f"input_emissions_{DATASET_UKNI}.csv"
)

if not ukni_raw.exists():
    print(f"  skip (not found): {ukni_raw.name}")
else:
    meta = datasets_df[datasets_df["dataset"] == DATASET_UKNI].iloc[0]
    country_name = meta["name"]

    print(f"  UKNI ({DATASET_UKNI}): {YEARS_UKNI}")
    try:
        ukni_df = pd.read_csv(ukni_raw, converters={"CRT_Category": str})
        ukni_source = sources.UKNISource()

        for record in ukni_source.fetch(DATASET_UKNI, YEARS_UKNI, ukni_df):
            record["dataset"] = DATASET_UKNI
            record["country_name"] = country_name
            record["analysis_year"] = ANALYSIS_YEAR_UKNI
            record["data_year"] = record["year"]
            all_rows.append(record)
    except Exception as exc:
        print(f"  Error: {exc}")

### 2.3 Preview

In [ ]:
combined = pd.DataFrame(all_rows)

print(f"Total rows   : {len(combined)}")
print(f"Countries    : {combined['country_name'].nunique()}")

# Sample: Germany 1990
combined[(combined["country_name"] == "Germany") & (combined["data_year"] == 1990)].head(10)

### 2.5 Initial save of intermediate CSVs
Save intermediate CSV files in data folder.

In [ ]:
writer = writers.IntermediateWriter(DATA_DIR)
saved: list[Path] = []

for (dataset, data_year), group in combined.groupby(["dataset", "data_year"]):
    analysis_year = int(group["analysis_year"].iloc[0])

    # Remove duplicates
    group_dedup = (
        group.sort_values(["category_code", "co2"])
        .drop_duplicates(subset=["category_code", "category"], keep="first")
        .reset_index(drop=True)
    )

    try:
        fpath = writer.write_dataframe(group_dedup, dataset, analysis_year, data_year)
        saved.append(fpath)
    except Exception as exc:
        print(f"  Error writing {dataset} {data_year}: {exc}")

print(f"Saved {len(saved)} files across {combined['dataset'].nunique()} datasets.")
for f in sorted(saved):
    print(f"  {f.relative_to(REPO_ROOT)}")

### 2.6 Verification of intermediate CSVs

In [ ]:
# ── Build expected set of (folder, year) pairs ────────────────────────────
expected: set[tuple[str, int]] = set()
for _, row in datasets_df.iterrows():
    ds = row["dataset"]
    if ds in SKIP_DATASETS:
        continue
    analysis_year = int(row["analysis_year"])
    fname = "nl" if ds.startswith("nl") else ds
    if ds in UNFCCC_EXCEL_DATASETS:
        for data_year, _ in UNFCCC_EXCEL_DATASETS[ds]:
            expected.add((fname, data_year))
    else:
        for yr in sorted({1990, analysis_year}):
            expected.add((fname, yr))

# ── Filter out UKNI files from validation ──────────────────────────────────
ukni_files = [f for f in saved if "UKNI01_northern_ireland" in str(f)]
other_files = [f for f in saved if "UKNI01_northern_ireland" not in str(f)]

if ukni_files:
    print(f"Skipping validation for UKNI ({len(ukni_files)} files)")

# File verification (excluding UKNI)
v_fails, v_warns = validators.verify_outputs(other_files, expected, DATA_DIR)
# Value validation (excluding UKNI)
d_fails, d_warns = validators.validate_outputs(other_files, DATA_DIR)

# Report warnings and failures
warns = v_warns + d_warns
fails = v_fails + d_fails

if warns:
    print(f"WARNINGS ({len(warns)}):")
    for w in warns:
        print(f"  ⚠  {w}")
    print()

if fails:
    print(f"FAILURES ({len(fails)}):")
    for f in fails:
        print(f"  ✗  {f}")
    raise AssertionError(f"{len(fails)} check failure(s) — see above.")
else:
    print(f"✓  No failures ({len(other_files)} files verified, {len(ukni_files)} UKNI files skipped).")

In [ ]:
# Verification of NL 1990 file consistency
nl2015_1990 = DATA_DIR / "nl" / "2015" / "15_emissions" / "intermediate_emissions_1990.csv"
nl2019_1990 = DATA_DIR / "nl" / "2019" / "15_emissions" / "intermediate_emissions_1990.csv"
nl2023_1990 = DATA_DIR / "nl" / "2023" / "15_emissions" / "intermediate_emissions_1990.csv"

if nl2015_1990.exists() and nl2019_1990.exists():
    df_2015 = pd.read_csv(nl2015_1990)
    df_2019 = pd.read_csv(nl2019_1990)
    if not df_2015.equals(df_2019):
        shutil.copy2(nl2019_1990, nl2015_1990)
        print(f"  ⚠  nl 1990 differed from nl2019 1990 — overwritten with nl2019 version")
    else:
        print(f"  ✓  nl and nl2019 intermediate_emissions_1990.csv are identical")
else:
    print(
        f"  skip: one or both NL 1990 files not found (2015: {nl2015_1990.exists()}, 2019: {nl2019_1990.exists()})"
    )

if nl2019_1990.exists():
    nl2023_1990.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(nl2019_1990, nl2023_1990)
    print(f"  ✓  Copied nl2019 intermediate_emissions_1990.csv → nl2023")
else:
    print(f"  skip: nl2019 intermediate_emissions_1990.csv not found")

### 2.7 Data fixes
See [Additional processing steps](README.md#additional-processing-steps) in the README for more information.

#### Sweden and Singapore data corrections

In [ ]:
# ── Singapore: 2.B.8.b = residual from 2.B.8 ─────────────────────────────
def _get_file_path(dataset: str, year: int) -> Path:
    """Helper to get intermediate file path."""
    ds_row = datasets_df[datasets_df["dataset"] == dataset]
    if ds_row.empty:
        return None
    analysis_year = int(ds_row.iloc[0]["analysis_year"])
    fname = "nl" if dataset.startswith("nl") else dataset
    suffix = str(year) if year == 1990 else "default"
    return (
        DATA_DIR / fname / str(analysis_year) / "15_emissions" / f"intermediate_emissions_{suffix}.csv"
    )


for dataset, year in [("SG_singapore", 1990), ("SG_singapore", 2019)]:
    fpath = _get_file_path(dataset, year)
    if not fpath or not fpath.exists():
        print(f"  skip (not found): {dataset} {year}")
        continue
    label = str(fpath.relative_to(DATA_DIR))
    df, val = processors.set_child_residual(pd.read_csv(fpath), "2.B.8", "2.B.8.b")
    if val is None:
        print(f"  –  {label}: 2.B.8.b — not applicable (skipped)")
    else:
        df.to_csv(fpath, index=False)
        print(f"  ✓  {label}: 2.B.8.b = {val:.1f} Gg CO₂  (residual from 2.B.8)")

# ── Sweden: 2.C.1.f = residual from 2.C.1, then 2.C.7 = residual from 2.C ──
SE_SWEDEN_PATH = _get_file_path("SE_sweden", 2019)
if SE_SWEDEN_PATH and SE_SWEDEN_PATH.exists():
    label = str(SE_SWEDEN_PATH.relative_to(DATA_DIR))
    df = pd.read_csv(SE_SWEDEN_PATH)
    df, val1 = processors.set_child_residual(df, "2.C.1", "2.C.1.f")
    df, val2 = processors.set_child_residual(df, "2.C", "2.C.7")
    if val1 is None or val2 is None:
        print(f"  –  {label}: Sweden 2.C fix — not applicable")
    else:
        df.to_csv(SE_SWEDEN_PATH, index=False)
        print(f"  ✓  {label}: 2.C.1.f = {val1:.1f}, 2.C.7 = {val2:.1f} Gg CO₂")
else:
    print(f"  skip (not found): SE_sweden 2019")

#### Reallocate mobile machinery - NL

In [ ]:
# ── NL mobile machinery reallocation ─────────────────────────────────────
for dataset in NL_DATASETS:
    ds_row = datasets_df[datasets_df["dataset"] == dataset]
    if ds_row.empty:
        print(f"  skip (not in config): {dataset}")
        continue
    analysis_year = int(ds_row.iloc[0]["analysis_year"])
    country_dir = DATA_DIR / "nl" / str(analysis_year) / "15_emissions"

    for suffix in ["default", "1990"]:
        src_path = country_dir / f"intermediate_emissions_{suffix}.csv"
        if not src_path.exists():
            print(f"  skip (not found): {src_path.name}")
            continue
        backup_path = country_dir / f"intermediate_emissions_{suffix}_before_mobile_machinery_reallocation.csv"
        src_path.rename(backup_path)
        df_reallocated = processors.reallocate_nl_mobile_machinery(pd.read_csv(backup_path))
        df_reallocated.to_csv(src_path, index=False)
        print(f"  ✓  {src_path.relative_to(REPO_ROOT)}")

## 3. Map CRT categories to ETM sectors

### 3.1 Map CRT categories to ETM sectors

In [ ]:
# ── Load configuration ────────────────────────────────────────────────────
crt_etm_mapping = pd.read_csv(TOOLS_EMISSIONS_INPUT_DIR / "crt_etm_mapping.csv")
ghg_config = pd.read_csv(TOOLS_EMISSIONS_INPUT_DIR / "config_etm_sector_include_ghg.csv")
INCLUDE_PARENTS_DATASETS = ["UKNI01_northern_ireland"]

# Initialize ETM mapper
etm_mapper = processors.ETMMapper(crt_etm_mapping, ghg_config)

# ── Process start year (default) with GHG filtering ────────────────────
print("Processing start year (default) emissions:")
mapped_results_default = {}
for fpath in sorted(DATA_DIR.rglob("15_emissions/intermediate_emissions_default.csv")):
    dataset = fpath.relative_to(DATA_DIR).parts[0]
    intermediate_df = pd.read_csv(fpath)

    is_nl = dataset in NL_DATASETS or dataset == "nl"
    include_parents = any(d in str(fpath) for d in INCLUDE_PARENTS_DATASETS)

    agg_df = etm_mapper.map_crt_to_etm(intermediate_df, dataset, include_parents, is_nl)
    agg_df = etm_mapper.split_lulucf(agg_df)
    long_df = etm_mapper.filter_ghg(agg_df)

    mapped_results_default[fpath] = long_df

print(f"  Processed {len(mapped_results_default)} datasets")

# ── Process 1990 base year with selective GHG filtering ───────────────────
# Keep all CO2 emissions but filter other_ghg based on config
print("\nProcessing 1990 base year emissions:")
mapped_results_1990 = {}
for fpath in sorted(DATA_DIR.rglob("15_emissions/intermediate_emissions_1990.csv")):
    dataset = fpath.relative_to(DATA_DIR).parts[0]
    intermediate_df = pd.read_csv(fpath)

    is_nl = dataset in NL_DATASETS or dataset == "nl"
    include_parents = any(d in str(fpath) for d in INCLUDE_PARENTS_DATASETS)

    agg_df = etm_mapper.map_crt_to_etm(intermediate_df, dataset, include_parents, is_nl)
    agg_df = etm_mapper.split_lulucf(agg_df)
    long_df = etm_mapper.filter_ghg(agg_df, always_include_co2=True)

    mapped_results_1990[fpath] = long_df

print(f"  Processed {len(mapped_results_1990)} datasets")

### 3.2 Combine both years

In [ ]:
# ── Combine both years  ─────────────────────────────────────
mapped_results_combined: dict[Path, dict[int, pd.DataFrame]] = {}

# Group by folder
default_by_folder = {fpath.parent: df for fpath, df in mapped_results_default.items()}
year1990_by_folder = {fpath.parent: df for fpath, df in mapped_results_1990.items()}

all_folders = set(list(default_by_folder.keys()) + list(year1990_by_folder.keys()))

for folder_path in all_folders:
    year_data = {}

    if folder_path in default_by_folder:
        # Get analysis year from folder path
        analysis_year = int(folder_path.parts[-2])
        year_data[analysis_year] = default_by_folder[folder_path]

    if folder_path in year1990_by_folder:
        year_data[1990] = year1990_by_folder[folder_path]

    if year_data:
        output_key = folder_path / "emissions.csv"
        mapped_results_combined[output_key] = year_data

print(f"Combined {len(mapped_results_combined)} datasets")

## 4. Export ETM output CSVs

In [ ]:
final_writer = writers.FinalWriter(DATA_DIR)
output_saved: list[Path] = []

for out_path, year_data in sorted(mapped_results_combined.items()):
    dataset = out_path.relative_to(DATA_DIR).parts[0]
    analysis_year = int(out_path.parts[-3])

    fpath = final_writer.write_long_format(dataset, analysis_year, year_data)
    output_saved.append(fpath)

print(f"Saved {len(output_saved)} emissions.csv output files.")

### 4.1 Generate Great Britain emissions

In [ ]:
# ── Generate GB emissions: GB = UK - UKNI ────────────────────────────────
uk_emissions_path = DATA_DIR / "UK_united_kingdom" / "2019" / "15_emissions" / "emissions.csv"
ukni_emissions_path = DATA_DIR / "UKNI01_northern_ireland" / "2018" / "15_emissions" / "emissions.csv"
gb_emissions_path = DATA_DIR / "GB_great_britain" / "2019" / "15_emissions" / "emissions.csv"

if uk_emissions_path.exists() and ukni_emissions_path.exists():
    stats = writers.calculate_gb_long_format(uk_emissions_path, ukni_emissions_path, gb_emissions_path)
    output_saved.append(gb_emissions_path)
    print(f"\n✓ Saved GB emissions")
    for year, year_stats in sorted(stats["per_year"].items()):
        print(f"  {year}:")
        print(f"    UK total:    {year_stats['uk_total']:.1f} kton")
        print(f"    UKNI total:  {year_stats['ukni_total']:.1f} kton")
        print(f"    GB total:    {year_stats['gb_total']:.1f} kton")
        print(f"    GB/UK ratio: {year_stats['gb_uk_ratio']:.2%}")
    if stats['negative_count'] > 0:
        print(f"  ⚠  {stats['negative_count']} negative values found:")
        for sector, subsector, use, ghg, year, value in stats['negative_rows']:
            print(f"       {year}  {sector} / {subsector} / {use} / {ghg}: {value:.2f} kton")
else:
    print("UK or UKNI emissions not found - skipping GB calculation")

### 4.2 Validate final 1990 emissions against source data

Validation approach:
- **API datasets**: Compare final emissions.csv against UNFCCC reported totals
- **Excel datasets**: Compare final emissions.csv against Summary2 sheet totals

Both validations exclude LULUCF, international transport, and indirect emissions.
Note that for `nl2023`, the 1990 emissions are 'copied' from the `nl2019`. Therefore, only the 1990 emissions for `nl2019` are validated here.

In [ ]:
# ── Validate 1990 totals against UNFCCC API source ───────────────────────
print("\nValidating 1990 emissions against UNFCCC reported totals...")
unfccc_fails, unfccc_warns = validators.validate_1990_unfccc_totals(
    output_saved, SOURCE_ANALYSES_DIR, UNFCCC_API_DATASETS
)

if unfccc_warns:
    print(f"\nWARNINGS ({len(unfccc_warns)}):")
    for w in unfccc_warns:
        print(f"  ⚠  {w}")
    print()
else:
    print("✓  All 1990 UNFCCC validations passed.")

In [ ]:
# ── Validate Excel datasets against Summary2 sheet ───────────────────────
print("\nValidating Excel route datasets (final emissions vs Summary2 sheet)...")
excel_fails, excel_warns = validators.validate_excel_final_emissions(
    output_saved, UNFCCC_EXCEL_DATASETS
)

if excel_warns:
    print(f"\nWARNINGS ({len(excel_warns)}):")
    for w in excel_warns:
        print(f"  ⚠  {w}")
    print()
else:
    print("✓  All Excel validations passed.")